# Deep Learning in Practice
## Homework 1: Problem 2

Elías Masquil

Nicolás Violante

## Regression Problem: House Prices Prediction
In this exercise, we will solve a regression problem with a neural network.

**Objective:** The goal is to predict the house selling prices .

**Dataset:**  A csv file with 1460 samples is provided (on the course webpage). Each example contains four input features. We will use 1000 examples as training set, 200 as validation set and the rest as test set.   
   * **Feature names**: OverallQual, YearBuilt, TotalBsmtSF, GrLivArea
   * **Target**: SalePrice

**NB:** new required libraries: `pandas`, `seaborn`.

In [ ]:
# Load the TensorBoard notebook extension
%load_ext tensorboard

In [ ]:
# Remove old logs
! rm -rf runs/

In [ ]:
! pip install pandas seaborn

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn

import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from torch.utils.tensorboard import SummaryWriter

import seaborn as sns

%matplotlib inline

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DATA_PATH = '/content/drive/My Drive/Colab Notebooks/data/house_prices.csv'

In [ ]:
# Load data:
df = pd.read_csv(DATA_PATH)

In [ ]:
df.head(3)

In [ ]:
df.info() # get more information

### Data Analysis
Before training, we need first to analyze the dataset, to know its properties better.

In [ ]:
sns.pairplot(df, x_vars=['OverallQual', 'YearBuilt', 'TotalBsmtSF', 'GrLivArea'], 
             y_vars=['SalePrice'])

### House prices prediction

Here is a skeleton of a neural network with a single layer (thus: a linear classifier). This is the model you'll start with and improve during this exercise.

Look at the code and run it to see its structure, then follow the questions below to iteratively improve the model.

In [ ]:
X = df[['OverallQual', 'YearBuilt', 'TotalBsmtSF', 'GrLivArea']] # get the four features from the dataframe
y = df['SalePrice'] # get the target values

In [ ]:
X_mean = X.iloc[:1000].mean(axis=0)
X_std = X.iloc[:1000].std(axis=0)

y_mean = y.iloc[:1000].mean(axis=0)
y_std = y.iloc[:1000].std(axis=0)

In [ ]:
X, y = (X - X_mean) / X_std, (y - y_mean) / y_std

If we don't normalize the data we get numerical inestabilities (e.g overflow on the gradients). While some shallow models could be trained using activations which clipped the outputs, using ReLU was impossible. Normalizing and standarizing the data allow us to test more models and avoid numerical errors.

In [ ]:
X_train = X.iloc[:1000]
y_train = y.iloc[:1000]

X_val = X.iloc[1000:1200]
y_val = y.iloc[1000:1200]

X_test = X.iloc[1200:]
y_test = y.iloc[1200:]

In [ ]:
# Construct a model with one layer
class Model(nn.Module):
    
    def __init__(self):
        super(Model, self).__init__()
        
        self.l1 = nn.Linear(4, 1)
        
    def forward(self, inputs):
        outputs = self.l1(inputs)
        return outputs

In [ ]:
# Define hyper-parameters:
model = Model()

# Choose the hyperparameters for training: 
num_epochs = 10
batch_size = 10

# Training criterion. This one is a mean squared error (MSE) loss between the output
# of the network and the target label
criterion = nn.MSELoss()

# Use SGD optimizer with a learning rate of 0.01
# It is initialized on our model
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

In [ ]:
train_set = TensorDataset(torch.from_numpy(np.array(X_train)).float(), 
                          torch.from_numpy(np.array(y_train)).float()) # creat the dataset.

In [ ]:
def train(num_epochs, batch_size, criterion, optimizer, model, dataset, eval_function=None, X_val=None, y_val=None, summary_writer=None):
    dataloader = DataLoader(dataset, batch_size, shuffle=True)
    mse_metric = nn.MSELoss()
    for epoch in range(num_epochs):
        model.train()
        epoch_average_loss = 0.0
        mse = 0.0
        for (X, y) in (dataloader):
            y_pre = model(X)
            if y_pre.shape[1] == 1:
                y_pre = y_pre.view(-1)
            else:
                with torch.no_grad():
                    mse += mse_metric(y_pre[:,0], torch.tensor(np.array(y)).float()).item() * batch_size / len(train_set)
            loss = criterion(y_pre, y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            epoch_average_loss += loss.item() * batch_size / len(train_set)
            
        if ((epoch+1)%1 == 0):
                print('Epoch [{}/{}], Loss_error: {:.4f}'
                      .format(epoch+1, num_epochs,  epoch_average_loss))
                if summary_writer:
                    summary_writer.add_scalar('train_loss', epoch_average_loss, epoch)
                    summary_writer.add_scalar('train_mse', mse, epoch)
                    eval(model, criterion, X_val, y_val, summary_writer, epoch)

In [ ]:
train(num_epochs, batch_size, criterion, optimizer, model, train_set)

#### Evaluate the Model on the validation set

In [ ]:
# Calculate mean squared error on validation set
model.eval()
with torch.no_grad():
    y_pre_val = model(torch.from_numpy(np.array(X_val)).float()).view(-1)
error = criterion(y_pre_val, torch.tensor(np.array(y_val)).float()).item()
print('The loss on validation set is:', error)

### Exercise 1: Impact of the architecture of the model

The class `Model` is the definition of your model. You can now modify it to try out different architectures and
see the impact of the following factors:

* Try to add more layers (1, 2, 3, more ?)
* Try different activation functions ([sigmoid](https://pytorch.org/docs/stable/nn.functional.html#torch.nn.functional.sigmoid), [tanh](https://pytorch.org/docs/stable/nn.functional.html#torch.nn.functional.tanh), [relu](https://pytorch.org/docs/stable/nn.functional.html#torch.nn.functional.relu), etc.)
* Try to change the number of neurons in each layer (5, 10, 20, more ?)

In [ ]:
class Model(nn.Module):
    
    def __init__(self, n_hidden=1, n_neurons=5, activation=nn.Sigmoid):
        super(Model, self).__init__()
        self.layers = nn.ModuleList()
        # Input layer
        self.layers.append(nn.Linear(4, n_neurons))
        # Hidden layers
        for n in range(n_hidden-1):
            self.layers.append(nn.Linear(n_neurons, n_neurons))
        # Output layer
        self.layers.append(nn.Linear(n_neurons, 1))

        self.activation = activation()
        

    def forward(self, inputs):
        x = inputs
        for layer in self.layers[:-1]:
            x = layer(x)
            x = self.activation(x)
        outputs = self.layers[-1](x)
        return outputs

In [ ]:
def eval(model, criterion, X_val, y_val, summary_writer=None, epoch=None):
    mse_metric = nn.MSELoss()
    mse = None
    # Calculate mean squared error on validation set
    model.eval()
    with torch.no_grad():
        y_pre_val = model(torch.from_numpy(np.array(X_val)).float())
        if y_pre_val.shape[1] == 1:
            y_pre_val = y_pre_val.view(-1)
        else:
            mse = mse_metric(y_pre_val[:,0], torch.tensor(np.array(y_val)).float()).item()
    error = criterion(y_pre_val, torch.tensor(np.array(y_val)).float()).item()
    if summary_writer:
        summary_writer.add_scalar("val_loss", error, epoch)
        if mse:
            summary_writer.add_scalar("val_mse", mse, epoch)
    else:
        print('The loss on validation set is:', error)
    return error

In [ ]:
best_error = np.inf
best_params = {}

for activation in [nn.Sigmoid, nn.ReLU, nn.Tanh]:
    for n_neurons in [5, 10, 20]:
        for n_hidden in range(1, 4):
            # Tensorboard stuff
            log_dir = "runs/ex1/" + f"n_hidden_{n_hidden}_n_neurons_{n_neurons}_activation_{activation}"
            summary_writer = SummaryWriter(log_dir)

            print("------------------")
            print(f"Number of hidden layers: {n_hidden} |  Number of neurons: {n_neurons} | Activation {activation}")
            # Define hyper-parameters:
            model = Model(n_hidden=n_hidden, n_neurons=n_neurons, activation=activation)

            # Use SGD optimizer with a learning rate of 0.01
            # It is initialized on our model
            optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

            # Choose the hyperparameters for training: 
            num_epochs = 10
            batch_size = 10

            criterion = nn.MSELoss()

            train(num_epochs, batch_size, criterion, optimizer, model, train_set, eval, X_val, y_val, summary_writer)

            error = eval(model, criterion, X_val, y_val)

            if error < best_error:
                best_error = error
                best_params = {
                    "n_hidden": n_hidden, 
                    "n_neurons": n_neurons,
                    "activation": activation
                    }

print("------------------")
print("Best params")
print(best_params)
print("Best error")
print(best_error)

In [ ]:
%tensorboard --logdir runs/ex1

* Activation function: sigmoid doesn't work when we stack layers (vanishing gradient) making the loss seem concave on the epochs. Tanh was more reliable than sigmoid, but ReLU achieves far better results consistently while being used with different sets of hyper-parameters.

* Number of layers and neurons: the best results in validation (as well as in training) are obtained when using 20 neurons and 1 and 2 hidden layers respectively. Using 20 neurons didn't lead to overfiting, and it was better than using less units. On the other hand, using 3 layers instead of 2 didn't really help. 

We'll continue the experiments with ReLU, 20 neurons and [1,2] hidden layers.

### Exercise 2: Impact of the optimizer

Retrain the model with different parameters of the optimizer; you can change then in the cell initializing the optimizer, after the definition of your model.

* Use different batch sizes, from 10 to 400 e.g.
* Try different values of the learning rate (between 0.001 and 10), and see how they impact the training process. Do all network architectures react the same way to different learning rates?
* Change the duration of the training by increasing the number of epochs
* Try other optimizers, such as [Adam](https://pytorch.org/docs/stable/optim.html?highlight=adam#torch.optim.Adam) or [RMSprop](https://pytorch.org/docs/stable/optim.html?highlight=rmsprop#torch.optim.RMSprop)

**Note:** These changes may interact with your previous choices of architectures, and you may need to change them as well!

Using a learning rate > 0.1 led to numerical errors on the gradients.

In [ ]:
best_error = np.inf
best_params = {}

for n_hidden in [1,2]:
    for batch_size in [10, 100, 400]:
        for learning_rate in [0.001, 0.01, 0.1]:
            for optimizer in [torch.optim.SGD, torch.optim.Adam]:
                # Tensorboard stuff
                log_dir = "runs/ex2/" + f"batch_size{batch_size}_learning_rate{learning_rate}_optimizer_{optimizer}_n_hidden{n_hidden}"
                summary_writer = SummaryWriter(log_dir)

                print("------------------")
                print(f"Batch size: {batch_size} |  Learning rate: {learning_rate} | Optimizer {optimizer} | Number of hidden layers {n_hidden}")
                # Define hyper-parameters:
                model = Model(n_hidden=n_hidden, n_neurons=20, activation=torch.nn.ReLU)

                optimizer = optimizer(model.parameters(), lr=learning_rate)

                # Choose the hyperparameters for training: 
                num_epochs = 10

                criterion = nn.MSELoss()

                train(num_epochs, batch_size, criterion, optimizer, model, train_set, eval, X_val, y_val, summary_writer)

                error = eval(model, criterion, X_val, y_val)

                if error < best_error:
                    best_error = error
                    best_params = {
                        "batch_size": batch_size, 
                        "learning_rate": learning_rate,
                        "optimizer": optimizer,
                        "n_hidden": n_hidden,
                        }

print("------------------")
print("Best params")
print(best_params)
print("Best error")
print(best_error)

In [ ]:
%tensorboard --logdir runs/ex2

Varying the batch size, optimizer, and learning rate, further improved the results of the model.

* Overall, ADAM led to faster convergence than SGD.

* A large batch size = 400 combined with a small learning rate 0.001 is far too slow and we're far from convergence. For getting fast convergence with a big batch size we should use a high learning rate, which can turn training unstable.

* Batch size 10 is usually quite small and the loss (and gradients) are very noisy overall. However for a sufficiently small LR and ADAM, this seems to work.

Aiming for the best but also most stable results, we'll continue with the following parameters: 2 hidden layers, batch size 100, learning rate 0.01, ADAM optimizer

In [ ]:
# Tensorboard stuff
log_dir = "runs/ex2_5/"
summary_writer = SummaryWriter(log_dir)

# Define hyper-parameters:
model = Model(n_hidden=2, n_neurons=20, activation=torch.nn.ReLU)

optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

# Choose the hyperparameters for training: 
num_epochs = 1000
batch_size = 100

criterion = nn.MSELoss()

train(num_epochs, batch_size, criterion, optimizer, model, train_set, eval, X_val, y_val, summary_writer)

error = eval(model, criterion, X_val, y_val)

In [ ]:
%tensorboard --logdir runs/ex2_5

After deciding the hyper-parameters we trained the model for 1000 epochs. We're getting better results than in the previous parts but after ~200 epochs it starts to overfit the training data a little, therefore we're using early stopping and training the model for 250 epochs.

### Exercise 3: Impact of the loss function
As mensioned before in the first problem (binary classification), one can minimize the negative of log-likelihood of the probability for all samples $x$: $$ \sum_{(x,y) \,\in\, \text{Dataset}} - \log p(y | x) $$ If we define $p(y_i | x_i) = \frac{1}{\sqrt{2\pi}}e^{-\frac{(y_i - f(x_i))^2}{2}}$, then the loss function becomes the mean squared error. 

There is another loss function worth to try: the Gaussian likelihood loss function. 
Rather than predicting a single value $y$ given $x$, we predict a probability distribution over possible answers, which helps dealing with ambiguous cases and expressing uncertainty. To do this, for each possible input $x$, the network will output the parameters of the distribution modeling $p(y|x)$. For instance in our case, we choose to model output distributions with Gaussian distributions $\mathcal{N}(\mu, \sigma)$, which are parameterized by their mean $\mu$ and their standard deviation $\sigma$. Therefore for each input $x$ we have to output two quantities: $\mu(x)$ and $\sigma(x)$. The probability becomes: $$p(y_i | x_i) = \frac{1}{\sqrt{2\pi \sigma(x_i)^2}}e^{-\frac{(y_i - \mu(x_i))^2}{2\sigma(x_i)^2}}$$ Then the loss function becomes: $$L =\sum\limits_{i=1}^{N}  \frac{1}{2} \log ( 2\pi\sigma_i^{2} ) + \frac{1}{2\sigma_i^{2}}  (y_{i} - \mu_i)^{2}$$ If we set $\sigma=1$, we obtain MSE the loss function. 

* Try to replace the loss function with this one, and compare the differences between the two losses.
 
* **Hints**: 
    * You need two outputs of your network, one represents the $\mu(x_i)$, another for $\log( \sigma(x_i)^2 )$ (better for optimization) 
    * Try deeper models, or you will not predict the variance $\sigma$ well. 


In [ ]:
def gaussian_loss(y_pred, y):
    mu_pred = y_pred[:,0]
    log_sigma_pred = y_pred[:,1]
    loss_var = (torch.log(torch.tensor(np.pi) * 2) + log_sigma_pred).mean()
    loss_mean = ((y - mu_pred) ** 2 / torch.exp(log_sigma_pred)).mean()
    return (0.5 * loss_var + 0.5 * loss_mean)


class GaussianModel(nn.Module):
    
    def __init__(self, n_hidden=1, n_neurons=5, activation=nn.Sigmoid):
        super(GaussianModel, self).__init__()
        self.layers = nn.ModuleList()
        # Input layer
        self.layers.append(nn.Linear(4, n_neurons))
        # Hidden layers
        for n in range(n_hidden-1):
            self.layers.append(nn.Linear(n_neurons, n_neurons))
        # Output layer
        self.layers.append(nn.Linear(n_neurons, 2))

        self.activation = activation()
        

    def forward(self, inputs):
        x = inputs
        for layer in self.layers[:-1]:
            x = layer(x)
            x = self.activation(x)
        return self.layers[-1](x)

In [ ]:
for n_hidden in [5, 10]:
    for n_neurons in [20, 40, 60]:
        # Tensorboard stuff
        log_dir = "runs/ex3/" + f"n_hidden{n_hidden}_n_neurons_{n_neurons}"
        summary_writer = SummaryWriter(log_dir)

        # Define hyper-parameters:
        model = GaussianModel(n_hidden=n_hidden, n_neurons=n_neurons, activation=torch.nn.ReLU)

        optimizer = torch.optim.Adam(model.parameters(), lr=0.00005)

        # Choose the hyperparameters for training: 
        num_epochs = 200
        batch_size = 25

        criterion = gaussian_loss

        train(num_epochs, batch_size, criterion, optimizer, model, train_set, eval, X_val, y_val, summary_writer);

        error = eval(model, criterion, X_val, y_val);

In [ ]:
%tensorboard --logdir runs/ex3

With the exact same parameters the loss was not decaying too much during training. After experimenting, we increase the number of hidden layers to 5 and the number of neurons to 60. It's interesting that altough the MSE in validation keeps decreasing, the loss might increase.

Final model: 5 hidden layers, 60 neurons per layer.

----

Finally we compare over the MSE in validation, both "best" models.

In [ ]:
# MSE model:
mse_model = Model(n_hidden=2, n_neurons=20, activation=torch.nn.ReLU)

optimizer = torch.optim.Adam(mse_model.parameters(), lr=0.01)

# Choose the hyperparameters for training: 
num_epochs = 250
batch_size = 100

criterion = nn.MSELoss()

train(num_epochs, batch_size, criterion, optimizer, mse_model, train_set)


mse_model.eval()
with torch.no_grad():
    y_pre_val = mse_model(torch.from_numpy(np.array(X_val)).float()).view(-1)
    error = criterion(y_pre_val, torch.tensor(np.array(y_val)).float()).item()
    unnormalized_error = criterion(y_pre_val*y_std+y_mean, torch.tensor(np.array(y_val*y_std+y_mean)).float()).item()

print(f"MSE model validation error: {error}")
print(f"MSE model validation RMSE: {np.sqrt(unnormalized_error)}")

In [ ]:
# Tensorboard stuff
log_dir = "runs/final_model"
summary_writer = SummaryWriter(log_dir)

# Gaussian likelihood model:
gaussian_model = GaussianModel(n_hidden=5, n_neurons=60, activation=torch.nn.ReLU)

optimizer = torch.optim.Adam(gaussian_model.parameters(), lr=0.00005, weight_decay=0.001)

# Choose the hyperparameters for training: 
num_epochs = 1000
batch_size = 100

criterion = gaussian_loss
test_loss = nn.MSELoss()

train(num_epochs, batch_size, criterion, optimizer, gaussian_model, train_set, eval, X_val, y_val, summary_writer)

gaussian_model.eval()

with torch.no_grad():
    y_pre_val = gaussian_model(torch.from_numpy(np.array(X_val)).float())
    error = test_loss(y_pre_val[:,0], torch.tensor(np.array(y_val)).float()).item()
    loss_in_val = criterion(y_pre_val, torch.tensor(np.array(y_val)).float()).item()

print(f"Gaussian likeliood Model Validation error: {error}")
print(f"Gaussian likelihood model loss in val:{loss_in_val}")

When training the best architecture longer, we observed some overfitting to the loss. Although the validation loss started to increase, the validation error kept decreasing -> the model was getting better price predictions but it estimate of the variance was getting worse. By adding weight decay we reduced the overfitting and now both the loss and the error decreased during training.

Final model (after playing with the learning rate, epochs and bs) 0.00005 learning rate, 100 batch size, 1000 epochs, Adam optimizer with weight decay of 0.001

In [ ]:
%tensorboard --logdir runs/final_model

### Exercice 4: Prediction on test set

* Once you have a model that seems satisfying on the validation dataset, you SHOULD evaluate it on a test dataset that has never been used before, to obtain a final accuracy value.
* When using the Gaussian likelihood function, the confidence of the network in its prediction is reflected in the variance it outputs. It can be interesting to check how this uncertainty varies with the data. For example, the uncertainty will decrease when the feature `OverallQual` increases. Plot the variance $\sigma(x)$ w.r.t one of the three features, on test set, and describe what you observe.

In [ ]:
mse_model.eval()

criterion = nn.MSELoss()

with torch.no_grad():
    y_pre_test = mse_model(torch.from_numpy(np.array(X_test)).float()).view(-1)
    error = criterion(y_pre_test, torch.tensor(np.array(y_test)).float()).item()
    unnormalized_error = criterion(y_pre_test*y_std+y_mean, torch.tensor(np.array(y_test*y_std+y_mean)).float()).item()
print(f"MSE model test error: {error}")
print(f"MSE model RMSE: {np.sqrt(unnormalized_error)}")

In [ ]:
gaussian_model.eval()

criterion = gaussian_loss

with torch.no_grad():
    y_pre_test = gaussian_model(torch.from_numpy(np.array(X_test)).float())
    error = test_loss(y_pre_test[:,0], torch.tensor(np.array(y_test)).float()).item()
    unnormalized_error = test_loss(y_pre_test[:,0]*y_std+y_mean, torch.tensor(np.array(y_test*y_std+y_mean)).float()).item()
    loss_in_test = criterion(y_pre_test, torch.tensor(np.array(y_test)).float()).item()
print(f"Gaussian likeliood model test error: {error}")
print(f"Gaussian likelihood model RMSE: {np.sqrt(unnormalized_error)}")
print(f"Gaussian likelihood model loss in test:{loss_in_test}")

In [ ]:
variance_test = torch.sqrt(torch.exp(y_pre_test[:,1]))

import matplotlib.pyplot as plt

X_test_unnormalized = X_test * X_std + X_mean

plt.plot(X_test_unnormalized["OverallQual"], variance_test * y_std, ".")
plt.xlabel("Overall Quality")
plt.ylabel("$\sigma(x_i)$")